## Imports


In [2]:
# Import Required Libraries
import polars as pl
from pathlib import Path

## Define Data Path


In [3]:
# Define the path to the data folder
data_folder = Path(
    "C:/Users/Mudit PC/OneDrive/Desktop/Data analytics projects/Indian_Beauty_Analytics/data")

## Load the datasets


In [4]:
# Load the customers dataset
customers_df = pl.read_csv(data_folder / "customers.csv")

# Load the products dataset
products_df = pl.read_csv(data_folder / "products.csv")

# Load the transactions dataset
transactions_df = pl.read_csv(data_folder / "transactions.csv")

## Inspecting the Datasets


In [5]:
# Display first rows of customers dataset
customers_df.head()

# Display first rows of products dataset
products_df.head()

# Display first rows of transactions dataset
transactions_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival
str,str,str,str,i64,str,str
"""T0""","""c1""","""p4""","""2024-09-23T00:00:00.000000""",399,"""Bhubaneswar""",null
"""T1""","""c948""","""p1""","""2024-11-12T00:00:00.000000""",499,"""Ghaziabad""",null
"""T2""","""c802""","""p6""","""2024-03-04T00:00:00.000000""",249,"""Serampore""",null
"""T3""","""c525""","""p9""","""2024-09-02T00:00:00.000000""",299,"""Ghaziabad""",null
"""T4""","""c488""","""p6""","""2024-08-06T00:00:00.000000""",249,"""Purnia""",null


## Check dataset structure


In [6]:
# Show schema of customers dataset
customers_df.schema

# Show schema of products dataset
products_df.schema

# Show schema of transactions dataset
transactions_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', String),
        ('price', Int64),
        ('city', String),
        ('festival', String)])

## Convert purchase date to datetime


In [7]:
# Convert purchase_date column to datetime format
transactions_df = transactions_df.with_columns(
    pl.col("purchase_date").str.to_datetime())

In [8]:
transactions_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', Datetime(time_unit='us', time_zone=None)),
        ('price', Int64),
        ('city', String),
        ('festival', String)])

## Check for null values


In [9]:
# check missing values in transactions dataset
transactions_df.null_count()

# check missing values in products dataset
products_df.null_count()

# check missing values in customers dataset
customers_df.null_count()

customer_id,name,city,age,gender
u32,u32,u32,u32,u32
0,0,0,0,0


## Validate dataset sizes


In [10]:
print(transactions_df.shape)

print(products_df.shape)

print(customers_df.shape)

(50000, 7)
(10, 4)
(1000, 5)


## Join product information to transactions

Transactions alone are not useful. we need product metadata


In [11]:
# Join transactions with products information
sales_df = transactions_df.join(
    products_df,
    on="product_id",
    how="left"
)

# Dropping the redundant columns
sales_df = sales_df.drop("price_right")

sales_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival,product_name,category
str,str,str,datetime[μs],i64,str,str,str,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null,"""Face Wash""","""Skincare"""
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null,"""Vitamin C Serum""","""Skincare"""
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null,"""Hair Oil""","""Haircare"""
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null,"""Face Pack""","""Skincare"""
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null,"""Hair Oil""","""Haircare"""


## Join customer information


In [12]:
# Join customer information
sales_df = sales_df.join(
    customers_df,
    on="customer_id",
    how="left"
)

# Dropping the redundant columns
sales_df = sales_df.drop("city_right")

sales_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival,product_name,category,name,age,gender
str,str,str,datetime[μs],i64,str,str,str,str,str,i64,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null,"""Face Wash""","""Skincare""","""Tanveer Kade""",43,"""Male"""
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null,"""Vitamin C Serum""","""Skincare""","""Christopher Rau""",50,"""Male"""
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null,"""Hair Oil""","""Haircare""","""Divya Krishnan""",30,"""Female"""
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null,"""Face Pack""","""Skincare""","""Jonathan Shroff""",54,"""Female"""
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null,"""Hair Oil""","""Haircare""","""Fariq Rai""",21,"""Male"""


## Create revenue column

Revenue is the core business metric


In [13]:
# Create revenue column based on product price
sales_df = sales_df.with_columns(
    pl.col("price").alias("revenue")
)

## Revenue by product


In [14]:
# Calculate revenue by products
sales_df.group_by("product_name").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

product_name,total_revenue
str,i64
"""Sunscreen SPF50""",3028544
"""Vitamin C Serum""",2466557
"""Face Wash""",2000985
"""Body Lotion""",1737671
"""Face Pack""",1580215
"""Aloe Vera Gel""",1503372
"""Hair Oil""",1210389
"""Nail Polish""",1041566
"""Herbal Shampoo""",980672


## Revenue by city


In [15]:
# Calculate revenue by city
sales_df.group_by("city").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

city,total_revenue
str,i64
"""South Dumdum""",144004
"""Berhampore""",138479
"""Allahabad""",137415
"""Ghaziabad""",133544
"""Haldia""",133489
…,…
"""Lucknow""",8619
"""Tumkur""",8224
"""Raiganj""",5634


## Revenue by festival


In [16]:
# Calculate revenue by fesitval

sales_df.group_by("festival").agg(
    pl.col("revenue").sum().alias("total_revenue")
).sort("total_revenue", descending=True)

festival,total_revenue
str,i64
null,13009013
"""Holi""",1097119
"""Diwali""",1088336
"""Summer""",1051482


## Check top selling products


In [17]:
# Count number of transcations per product

sales_df.group_by("product_name").count().sort("count", descending=True)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_14736\3571595247.py:3: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  sales_df.group_by("product_name").count().sort("count", descending=True)


product_name,count
str,u32
"""Face Pack""",5285
"""Nail Polish""",5234
"""Sunscreen SPF50""",5056
"""Aloe Vera Gel""",5028
"""Face Wash""",5015
"""Body Lotion""",4979
"""Vitamin C Serum""",4943
"""Herbal Shampoo""",4928
"""Hair Oil""",4861


## Reordering for better readability


In [18]:
# Reorder columns for cleaner analytics structure
sales_df = sales_df.select([
    "transaction_id",
    "customer_id",
    "product_id",
    "purchase_date",
    "festival",
    "city",
    "product_name",
    "category",
    "name",
    "age",
    "gender",
    "price",
    "revenue"
])

In [19]:
# Renaming name column for better readability
sales_df = sales_df.rename({
    "name": "customer_name"
})

In [20]:
sales_df.select(pl.col("transaction_id").n_unique())

transaction_id
u32
50000


In [21]:
sales_df.schema

Schema([('transaction_id', String),
        ('customer_id', String),
        ('product_id', String),
        ('purchase_date', Datetime(time_unit='us', time_zone=None)),
        ('festival', String),
        ('city', String),
        ('product_name', String),
        ('category', String),
        ('customer_name', String),
        ('age', Int64),
        ('gender', String),
        ('price', Int64),
        ('revenue', Int64)])

In [22]:
sales_df = sales_df.sort("purchase_date")

## Saving the data set in csv file


In [23]:
# Save cleaned sales dataset
sales_df.write_csv(data_folder / "sales_cleaned.csv")

## Saving the data set in parquet


In [24]:
sales_df.write_parquet(data_folder / "sales_cleaned.parquet")